In [ ]:
import json
def content_to_str(path,dictionary):
    words=[]
    with open(path, mode="r") as f: 
       jsonn = json.loads(f.read())
       for info in jsonn["content"]:
           for word in info:
               if word['word'] in dictionary:
                   words.append(word['word'])
    return words

In [ ]:
import gensim
from gensim import corpora
from random import random
def trainLDA(corpus,dictionary,num_topics):
    lda_model = gensim.models.ldamodel.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        passes=10,
        alpha='auto',
        random_state=int(random()*2147483647),
        per_word_topics=True
    )
    return lda_model

In [ ]:
def aaaaa(garbage,lsa_models):
    print("???????????????????????????????????????????????????????????")
    import json

    dictionary_path = "files/dictionary2.txt"
    with open(dictionary_path, mode="r") as f: 
        jsonn = json.loads(f.read())



    dictionary = {}
    for i in range(len(jsonn)):
        if jsonn[i][0] not in garbage:
            dictionary[jsonn[i][0]] = i
    #print(dictionary)


    # In[2]:


    paths=["../web crawler/files/ckip-topics/nchu-topic/",
           "../web crawler/files/ckip-topics/nkust-topic/",
           "../web crawler/files/ckip-topics/ntnu-topic/",
           "../web crawler/files/ckip-topics/nuk-topic/",
          ]
    from os import listdir
    topic_paths=[]
    for path in paths:
        for file in listdir(path):
            topic_paths.append(path + file) 
    topic_paths = topic_paths[:]



    from tqdm import tqdm
    import multiprocessing as mp

    pool = mp.Pool()
    jobs = []
    for path in topic_paths:
        job = pool.apply_async(content_to_str, (path,dictionary,))
        jobs.append(job)
    topics=[]
    for job in tqdm(jobs):    
        topics.append(job.get())
    pool.close()
    pool.join()


    # In[5]:


    from gensim import corpora

    # Tokenize the documents and remove stop words
    tokenized_docs = topics
    # Create a dictionary and convert the tokenized documents to bag-of-words vectors
    dictionary = corpora.Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    # Save the pre-processed corpus and dictionary to disk
    #corpora.MmCorpus.serialize('corpus.mm', corpus)
    #dictionary.save('dictionary.dict')


    # In[6]:


    num_topics = 9
    num_models = 10


    # In[7]:




    # Load the corpus and dictionary
    #corpus = corpora.MmCorpus('corpus.mm')
    #dictionary = corpora.Dictionary.load('dictionary.dict')



    pool = mp.Pool()
    jobs = []
    for _ in range(num_models):
        job = pool.apply_async(trainLDA, (corpus,dictionary,num_topics,))
        jobs.append(job)
    lda_models=[]
    for job in tqdm(jobs):    
        lda_models.append(job.get())
    pool.close()
    pool.join()
    #lda_model.save('lda_model3')

    for lda_model in lda_models:
        print(id(lda_model))
        # Print the topics and their top words
        for topic_id, topic in lda_model.show_topics(num_topics=num_topics, num_words=10, formatted=False):
            print('Topic {}: {}'.format(topic_id, [word for word, _ in topic]))
    # ## evaluate each model

    # In[8]:


    topics_models = [[0 for _2 in range(num_models)] for _1 in range(len(topics))]
    for c in range(num_models):
        lda_model = lda_models[c]
        for c1 in range(len(topics)):
            # Convert the pre-processed paragraph to a bag-of-words vector
            bow_vector = dictionary.doc2bow(topics[c1])
            # Infer the topic distribution of the paragraph using the LDA model
            topic_distribution = lda_model.get_document_topics(bow_vector)
            maxjp=(0,0)
            for j, p in topic_distribution:
                if maxjp[1] < p:
                    maxjp = (j,p)
            topics_models[c1][c] = maxjp[0]# or append 


    # ## using evaluate result to find out good model

    # In[9]:


    good_model = {}
    for i in tqdm(range(len(topics_models))):
        for j in range(len(topics_models)):
            #compare two topics
            if i >= j:
                #skip repeat
                continue
            diff = 0
            for k in range(len(topics_models[i])):    
                if topics_models[i][k] == topics_models[j][k]:
                    diff += 1
            if diff > len(topics_models[i]) * 0.8:#two topics are similar
                #encode topic model serialize
                code = 0
                for l in range(len(topics_models[i])):
                    code *= num_topics+1
                    if topics_models[i][l] == topics_models[j][l]:
                        code += topics_models[i][l] + 1 # 0 mean no model match
                if code in good_model:
                    good_model[code] += 1
                else:
                    good_model[code] = 1


    # ## select top 10%

    # In[10]:


    good_model = sorted(good_model.items(), key=lambda item: item[1],reverse=True)
    #print(good_model)

    i = 0
    for i in range(len(good_model)):
        if good_model[i][1] < len(topics)/10:
            break
    good_model = good_model[:i]

    #good_model = good_model[:int(len(good_model) * 0.1)]
    #print(good_model)


    # ## remove similar solution and show

    # In[11]:


    topics_key_words = []
    for lda_model in lda_models:
        topics_key_words.append(lda_model.show_topics(num_topics=num_topics, num_words=10, formatted=False))
    sol = []
    for i, _ in good_model:
        a = [0 for _ in range(num_models)]
        tmp = i
        c = num_models - 1
        while tmp != 0:
            a[c] = (tmp % (num_topics + 1) - 1)
            tmp //= (num_topics + 1)
            c -= 1
        #remove similar solution
        for s in sol:
            d = len(s)
            for c in range(len(s)):
                if a[c] == s[c]:
                    d -= 1
            if d < len(a) * 0.8:
                break
        else:
            sol.append(a)   


    model_topics = [] 
    for s in sol:
        tmp = []
        m = 0
        for c in range(len(s)):
            if s[c] > 0:
                m += 1

        for i, _topic in enumerate(topics_models):
            a = 0
            for c in range(len(s)):
                if s[c] == _topic[c]:
                    a += 1
            if a > m * 0.7:
                tmp.append(i)
        model_topics.append(tmp)

    #競賽、志工、實習、文化、環保、外語


    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    # 使用TfidfVectorizer將文本轉化為詞頻矩陣
    vectorizer = TfidfVectorizer(max_df=0.5, min_df=10, stop_words='english')
    lsa_results = lsa_models[:]
    for c in range(len(model_topics)):
        _topics = []
        for i in model_topics[c]:
            tmp = ""
            for w in topics[i]:
                tmp += w + " "
            _topics.append(tmp)
        print(len(_topics))
        tfidf_matrix = vectorizer.fit_transform(_topics)
        # 使用TruncatedSVD進行LSA分析
        lsa = TruncatedSVD(n_components=1)
        lsa_matrix = lsa.fit_transform(tfidf_matrix)

        # 打印前10個詞彙和概念
        terms = vectorizer.get_feature_names_out()
        for comp in lsa.components_:
            terms_comp = zip(terms, comp)
            sorted_terms = sorted(terms_comp, key=lambda x:x[1], reverse=True)
            tmp = {}
            for keyword, t in sorted_terms:
                if t < 0.039:
                    break
                if t > 0.13:
                    #dictionary.pop(keyword)
                    garbage.append(keyword)
                    continue
                tmp[keyword] = t
            if len(tmp) > 9:
                lsa_results.append(tmp)

    import numpy as np
    t = [0 for _ in range(len(lsa_results))]
    sums = [0 for _ in range(len(lsa_results))]
    docs = []
    for topic_indexs in model_topics:
        for index in topic_indexs:
            docs.append(topics[index])
    for doc in docs:
        p = [0 for _ in range(len(lsa_results))]
        for word in doc:
            for i, _lsa in enumerate(lsa_results):
                if word in _lsa:
                    p[i] += _lsa[word]
        max_index = np.argmax(p)
        if p[max_index] > 0.39:
            t[max_index] += 1
            sums[max_index] += p[max_index]        
    topicsum = sum(t)   
    print(t)
    print(sum(t))
    return topicsum,lsa_results



In [ ]:
garbage = ["張貼","活動組","活動","聯絡","建工","方式","教師","期間","附件","小姐",
           "分機","大樓",'聯絡人','網址','地點','老師','場次','內容','職員','消息',
           '學期','人員','規定','口罩','疫情','同學','日期','事項','小時','詳情']
lsa_models = []
topicsum = 0
import time
while topicsum < 5000:
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    topicsum,lsa_results = aaaaa(garbage,lsa_models)
    print(topicsum,' ',len(lsa_results))
    for result in lsa_results:
        lsa_models.append(result)
        for word in result.keys():
            garbage.append(word)
print(llsa_models())
